# Gradient Descent from Scratch
Simple linear regression, $\hat{y} = \theta_0 x + \theta_1$, fitted with vanilla gradient descent on the loss $\frac{1}{2}\,\text{mean}\big((\hat y - y)^2\big)$.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
np.random.seed(42)

### Load data
Uses `Synthetic.csv` if it is in the same folder; otherwise generates similar data.

In [ ]:
if os.path.exists("Synthetic.csv"):
    df = pd.read_csv("Synthetic.csv").drop(columns=["Unnamed: 0"], errors="ignore")
else:
    x_raw = np.random.uniform(0, 100, 50)
    df = pd.DataFrame({"X": x_raw, "Y": 3 * x_raw + 5 + np.random.normal(0, 10, 50)})
df.head()

### Prepare data
Standardising `x` lets us use a much larger learning rate and converge far faster.

In [ ]:
x_raw, y = df.iloc[:, 0].values, df.iloc[:, 1].values
x = (x_raw - x_raw.mean()) / x_raw.std()

### Gradient descent
Gradients (m = number of samples):  
$\partial L/\partial\theta_0 = \frac{1}{m}\sum (\hat y - y)\,x$,  $\partial L/\partial\theta_1 = \frac{1}{m}\sum (\hat y - y)$

In [ ]:
def gradient_descent(x, y, lr=0.1, epochs=10000, tol=1e-6):
    t0, t1 = np.random.randn(2)           # slope, intercept
    losses = []
    for _ in range(epochs):
        err = t0 * x + t1 - y
        losses.append(0.5 * np.mean(err ** 2))
        t0 -= lr * np.mean(err * x)
        t1 -= lr * np.mean(err)
        if len(losses) > 1 and abs(losses[-2] - losses[-1]) < tol:
            break
    return t0, t1, losses

t0, t1, losses = gradient_descent(x, y)
print(f"theta_0 (slope) = {t0:.3f}, theta_1 (intercept) = {t1:.3f}")
print(f"Stopped after {len(losses)} epochs, final loss = {losses[-1]:.3f}")

### Results

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(losses); ax[0].set(xlabel="Epoch", ylabel="Loss (MSE/2)", title="Convergence")
ax[1].scatter(x, y, s=20); ax[1].plot(np.sort(x), t0 * np.sort(x) + t1, "r")
ax[1].set(xlabel="x (standardised)", ylabel="y", title="Fitted line")
plt.tight_layout(); plt.show()

### Sanity check against NumPy's closed-form fit

In [ ]:
slope, intercept = np.polyfit(x, y, 1)
print(f"np.polyfit -> slope = {slope:.3f}, intercept = {intercept:.3f}")